# Learning LSTM Autoencoder and LSTM Network on a simple Multivariate Timeseries Toy example

In [ ]:
# lstm autoencoder to recreate a timeseries
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Dropout

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from pylab import rcParams

def temporalize(X, y, lookback):
    '''
    A UDF to convert input data into 3-D
    array as required for LSTM network.
    '''
    Xs, ys = [], []
    for i in range(len(X) - lookback - 1):
        t = []
        for j in range(1, lookback + 1):
            # Collect past records up to the lookback period
            t.append(X[[(i + j + 1)], :])
        Xs.append(t)
        ys.append(y[i + lookback + 1])
    return Xs, ys

def flatten(X):
    '''
    Flatten a 3D array. Inverse of temporalize function
    
    Input
    X            A 3D array for lstm, where the array is sample x timesteps x features.
    
    Output
    flattened_X  A 2D array, sample x features.
    '''
    flattened_X = np.empty((X.shape[0], X.shape[2]))  # sample x features array.
    for i in range(X.shape[0]):
        flattened_X[i] = X[i, (X.shape[1]-1), :]
    return(flattened_X)

def scale(X, scaler):
    '''
    Scale 3D array.

    Inputs
    X            A 3D array for LSTM, where the array is sample x timesteps x features.
    scaler       A scaler object, e.g., sklearn.preprocessing.StandardScaler, sklearn.preprocessing.normalize
    
    Output
    X            Scaled 3D array.
    '''
    for i in range(X.shape[0]):
        X[i, :, :] = scaler.transform(X[i, :, :])
        
    return X

In [ ]:
# define input timeseries
# Import training data
dataset_train_raw = pd.read_csv("../data/raw/energy-informatics-2020/csvDataFeaturesTrain.csv", delimiter=';')
dataset_test_raw= pd.read_csv("../data/raw/energy-informatics-2020/csvDataFeaturesTest.csv", delimiter=';')

print(dataset_train_raw.columns)

#extract data and labels and create smaller subset
data_with_labels = dataset_train_raw.to_numpy()
cols = ['frameLen', 'ipTtl', 'tcpWinSize', 'tcpPduSize', 'attack']
dataset_train = dataset_train_raw[cols]
dataset_test = dataset_test_raw[cols]

In [ ]:
dataset_test[dataset_test['attack']==0]

In [ ]:
#timeseries = np.array([[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], [0.1**3, 0.2**3, 0.3**3, 0.4**3, 0.5**3, 0.6**3, 0.7**3, 0.8**3, 0.9**3]]).transpose()
num_samples = 1000

train = dataset_train.iloc[:num_samples, dataset_train.columns != 'attack'].values
y_train = dataset_train.iloc[:num_samples, dataset_train.columns == 'attack'].values

test = dataset_test.iloc[:, dataset_test.columns != 'attack'].values
y_test = dataset_test.iloc[:, dataset_test.columns == 'attack'].values

print(train.shape, y_train.shape)
print(test.shape, y_test.shape)

In [ ]:
timesteps = 5
n_features = train.shape[1] # number of features

X_train, y_train = temporalize(X = train, y = y_train, lookback = timesteps)
X_train = np.array(X_train)

X_test, y_test = temporalize(X = test, y = y_test, lookback = timesteps)
X_test = np.array(X_test)

X_train = X_train.reshape(X_train.shape[0], timesteps, n_features)
X_test = X_test.reshape(X_test.shape[0], timesteps, n_features)

print(X_train.shape, len(y_train))
print(X_test.shape, len(y_test))

In [ ]:
# define model
model = Sequential()
model.add(LSTM(256, activation='relu', input_shape=(timesteps, n_features), return_sequences=True))
model.add(LSTM(128, activation='relu', return_sequences=True))
model.add(LSTM(64, activation='relu', return_sequences=False))
model.add(RepeatVector(timesteps))
model.add(LSTM(64, activation='relu', return_sequences=True))
model.add(LSTM(128, activation='relu', return_sequences=True))
model.add(LSTM(128, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(n_features)))
model.compile(optimizer='adam', metrics=['accuracy'], loss='mse')
model.summary()

In [ ]:
#simple LSTM autoencoder
#the goal of this lstm network is to encode the sequence of 28 image rows in a single 2d latent vector, and decode (reconstruct) the
#row sequence with the decoder part

#input sequences of length 28 with 28 entries
#input_layer = tf.keras.layers.Input(shape=(timesteps, n_features))
#layer = input_layer

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

es = EarlyStopping(monitor="val_loss", min_delta=1e-10, patience=10, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1)
mcp = ModelCheckpoint(filepath='0.6-weights.h5', verbose=1, save_best_only=True, save_weights_only=True)

tb = TensorBoard(log_dir='./logs',
                histogram_freq=0,
                write_graph=True,
                write_images=True)

In [ ]:
timesteps =  X_train.shape[1] # equal to the lookback
n_features =  X_train.shape[2] # 59

epochs = 200
batch = 64
lr = 0.0001

lstm_autoencoder = Sequential()
# Encoder
lstm_autoencoder.add(LSTM(32, activation='relu', input_shape=(timesteps, n_features), return_sequences=True))
lstm_autoencoder.add(LSTM(16, activation='relu', return_sequences=False))
lstm_autoencoder.add(RepeatVector(timesteps))
# Decoder
lstm_autoencoder.add(LSTM(16, activation='relu', return_sequences=True))
lstm_autoencoder.add(LSTM(32, activation='relu', return_sequences=True))
lstm_autoencoder.add(TimeDistributed(Dense(n_features)))
lstm_autoencoder.compile(optimizer='adam', loss='mse')
lstm_autoencoder.summary()

cp = ModelCheckpoint(filepath="lstm_autoencoder_classifier.h5",
                               save_best_only=True,
                               verbose=0)

tb = TensorBoard(log_dir='./logs',
                histogram_freq=0,
                write_graph=True,
                write_images=True)

history = lstm_autoencoder.fit(X_train, X_train, 
                                                epochs=epochs, 
                                                batch_size=batch, 
                                                validation_data=(X_test, X_test),
                                                verbose=2).history

In [ ]:
# fit model
history = lstm_autoencoder.fit(
    X_train, X_train, 
    epochs=100, 
    batch_size=128,
    validation_split=0.2, 
    shuffle=True,
    callbacks=[mcp, tb, es, rlr],
    verbose=1).history

In [ ]:
fig, ax = plt.subplots()
plt.plot(history['loss'], linewidth=2, label='Train')
plt.plot(history['val_loss'], linewidth=2, label='Valid')
plt.legend(loc='upper right')
plt.title('Model loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# plot accuracy during training
plt.title('Model accuracy')
plt.plot(history['accuracy'], label='train')
plt.plot(history['val_accuracy'], label='test')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_curve
from sklearn.metrics import recall_score, classification_report, auc, roc_curve
from sklearn.metrics import precision_recall_fscore_support, f1_score

valid_x_predictions = lstm_autoencoder.predict(X_test)

In [ ]:
mse = np.mean(np.power(flatten(X_test) - flatten(valid_x_predictions), 2), axis=1)

error_df = pd.DataFrame({'Reconstruction_error': mse, 'True_class': np.array(y_test).flatten()})

precision_rt, recall_rt, threshold_rt = precision_recall_curve(error_df.True_class, error_df.Reconstruction_error)
plt.plot(threshold_rt, precision_rt[1:], label="Precision",linewidth=5)
plt.plot(threshold_rt, recall_rt[1:], label="Recall",linewidth=5)
plt.title('Precision and recall for different threshold values')
plt.xlabel('Threshold')
plt.ylabel('Precision/Recall')
plt.legend()
plt.show()

In [ ]:
threshold_fixed = 0.4
groups = error_df.groupby('True_class')
fig, ax = plt.subplots()

for name, group in groups:
    ax.plot(group.index, group.Reconstruction_error, marker='o', ms=3.5, linestyle='',
            label="Normal" if name == 0 else "Anomaly")
ax.hlines(threshold_fixed, ax.get_xlim()[0], ax.get_xlim()[1], colors="r", zorder=100, label='Threshold')
ax.legend()
plt.title("Reconstruction error for different classes")
plt.ylabel("Reconstruction error")
plt.xlabel("Data point index")
plt.show();

In [ ]:
LABELS = ["Normal","Anomaly"]

pred_y = [1 if e > threshold_fixed else 0 for e in error_df.Reconstruction_error.values]
conf_matrix = confusion_matrix(error_df.True_class, pred_y)

plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, xticklabels=LABELS, yticklabels=LABELS, annot=True, fmt="d");
plt.title("Confusion matrix")
plt.ylabel('True class')
plt.xlabel('Predicted class')
plt.show()

In [ ]:
false_pos_rate, true_pos_rate, thresholds = roc_curve(error_df.True_class, error_df.Reconstruction_error)
roc_auc = auc(false_pos_rate, true_pos_rate,)

plt.plot(false_pos_rate, true_pos_rate, linewidth=5, label='AUC = %0.3f'% roc_auc)
plt.plot([0,1],[0,1], linewidth=5)

plt.xlim([-0.01, 1])
plt.ylim([0, 1.01])
plt.legend(loc='lower right')
plt.title('Receiver operating characteristic curve (ROC)')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
# demonstrate reconstruction
X_pred = model.predict(X_train, verbose=0)

print('---Predicted---')
print(np.round(X_pred, 3))
print('---Actual---')
print(np.round(X_train, 3))

In [ ]:
loss, acc = model.evaluate(X_test, X_test)
print("Test set loss = ", loss)
print("Test set accuracy = ", acc)

In [ ]:
train_mae_loss = np.mean(np.abs(X_train - X_pred), axis=1)

plt.hist(train_mae_loss, bins=50)
plt.xlabel("Train MAE loss")
plt.ylabel("No of samples")
plt.show()

# Get reconstruction loss threshold.
threshold = np.max(train_mae_loss)
print("Reconstruction error threshold: ", train_mae_loss)
print("Reconstruction error threshold: ", threshold)

In [ ]:
# Checking how the first sequence is learnt
plt.plot(X_train[2])
plt.plot(X_pred[2])
plt.show()

In [ ]:
# Get test MAE loss.
x_test_pred = model.predict(X_test)
test_mae_loss = np.mean(np.abs(x_test_pred - X_test), axis=1)
test_mae_loss = test_mae_loss.reshape((-1))

plt.hist(test_mae_loss, bins=50)
plt.xlabel("test MAE loss")
plt.ylabel("No of samples")
plt.show()

# Detect all the samples which are anomalies.
anomalies = test_mae_loss > threshold
print("Number of anomaly samples: ", np.sum(anomalies))
print("Indices of anomaly samples: ", np.where(anomalies))
print(np.sum(anomalies)/X_test.shape[0]*100, "% are anomalies")

In [ ]:
#dataset_test_raw

In [ ]:
from sklearn.metrics.pairwise import paired_distances

#now reconstruct known data and reconstruct data that was not used during training
X_train_reconstructed = model.predict(X_train)
X_test_reconstructed = model.predict(X_test)

#calculate reconstruction errors (should be much lower for data which was used during training)
known_distances = paired_distances(X_train.reshape(X_train.shape[0], -1), X_train_reconstructed.reshape(X_train_reconstructed.shape[0], -1))
mean_distance_known = np.mean(known_distances)
std_known = np.std(known_distances)

In [ ]:
#print mean/std or reconstruction errors for known data
print("KNOWN: mean reconstruction error         : " + str(mean_distance_known))
print("KNOWN: std of reconstruction error       : " + str(std_known))

#set a simple anomaly threshold, all below that threshold is consideres as normal, all above is consideres as anomaly
anomaly_threshold = mean_distance_known + std_known
print("Anomaly Threshold                        : " + str(anomaly_threshold))

#calc mean distances for examples that were not present during training
unknown_distances = paired_distances(X_test.reshape(X_test.shape[0], -1), X_test_reconstructed.reshape(X_test_reconstructed.shape[0], -1))
mean_distance_unknown = np.mean(unknown_distances)
mean_std_unknown = np.std(unknown_distances)
print("UNKNOWN: mean reconstruction error       : " + str(mean_distance_unknown))
print("UNKNOWN: std of reconstruction error     : " + str(mean_std_unknown))

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_curve
from sklearn.metrics import recall_score, classification_report, auc, roc_curve
from sklearn.metrics import precision_recall_fscore_support, f1_score

mse = np.mean(np.power(flatten(X_test) - flatten(X_test_reconstructed), 2), axis=1)
error_df = pd.DataFrame({'Reconstruction_error': mse,
                        'True_class': y_test})

In [ ]:
precision_rt, recall_rt, threshold_rt = precision_recall_curve(error_df.True_class, error_df.Reconstruction_error)

In [ ]:
plt.plot(threshold_rt, precision_rt[1:], label="Precision",linewidth=5)
plt.plot(threshold_rt, recall_rt[1:], label="Recall",linewidth=5)
plt.title('Precision and recall for different threshold values')
plt.xlabel('Threshold')
plt.ylabel('Precision/Recall')
plt.legend()
plt.show()